<a href="https://colab.research.google.com/github/PranjalKabra/Fairness-Movie-Recommender/blob/main/Notebook_movie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import os
import zipfile
import math
from sklearn.metrics import accuracy_score, r2_score

In [12]:
print(tf.__version__)

2.20.0


## Unzip the data

In [13]:
zip_path = '/content/dataset_movie.zip'
extract_path = '/content/extracted_data'

if os.path.exists(zip_path):
  if(zipfile.is_zipfile(zip_path)):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
      zip_ref.extractall(extract_path)
      ratings_file = os.path.join(extract_path, "u.data")
      user_attribute_file = os.path.join(extract_path, "u.user")
      df1 = pd.read_csv(ratings_file, sep='\t', names=["userId","movieId","rating","timestamp"], engine='python')

      df1['count'] = df1.groupby('userId')['movieId'].transform('count')
      df1 = df1[df1['count'] > 300]
      # attribute file will contain the demographic data of the user
      df2 = pd.read_csv(user_attribute_file, sep='|', names=["userId","age","gender","occupation","dn"], engine='python')



In [14]:
print("Ratings Head:\n", df1.head())
print("\nUsers Head:\n", df2.head())

Ratings Head:
     userId  movieId  rating  timestamp  count
15     303      785       3  879485318    484
17     194      274       2  879539794    305
19     234     1184       2  892079237    480
24     308        1       4  887736532    397
36     181     1081       1  878962623    435

Users Head:
    userId  age gender  occupation     dn
0       1   24      M  technician  85711
1       2   53      F       other  94043
2       3   23      M      writer  32067
3       4   24      M  technician  43537
4       5   33      F       other  15213


## Data Cleaning


*   Female - 1, Male - 0
*   Don't use age for now
*   Drop time stamp too
*   Merge based on userId





In [15]:
df2['gender'] = df2['gender'].map({'F' : 1, 'M' : 0})
df3 = df2.drop(['age', 'occupation', 'dn'], axis = 1)
print("Cleaned users\n", df3.head())

df4 = df1.drop(['timestamp'], axis = 1)
df = pd.merge(df4, df3, on='userId', how='left')
print("\nMerged Dataset:\n", df.head())


Cleaned users
    userId  gender
0       1       0
1       2       1
2       3       0
3       4       0
4       5       1

Merged Dataset:
    userId  movieId  rating  count  gender
0     303      785       3    484       0
1     194      274       2    305       0
2     234     1184       2    480       0
3     308        1       4    397       0
4     181     1081       1    435       0


## Encoding Data for Embeddings

In [16]:
user_ids = df["userId"].unique().tolist()
movie_ids = df["movieId"].unique().tolist()

user2user_encoded = {x: i for i, x in enumerate(user_ids)}
userencoded2user = {i: x for i, x in enumerate(user_ids)} # For decoding later
movie2movie_encoded = {x: i for i, x in enumerate(movie_ids)}
movie_encoded2movie = {i: x for i, x in enumerate(movie_ids)} # For decoding later

# 3. Apply the mappings to create new, clean 'user' and 'movie' columns
df["user"] = df["userId"].map(user2user_encoded)
df["movie"] = df["movieId"].map(movie2movie_encoded)

In [17]:
num_users = len(user2user_encoded)
num_movies = len(movie_encoded2movie)
print(num_users, num_movies)

53 1595


In [18]:
# 5. Make sure ratings are floats (decimals) for mathematical stability
df["rating"] = df["rating"].values.astype(np.float32)

# 6. Find the min and max ratings
min_rating = min(df["rating"])
max_rating = max(df["rating"])

print(
    "Number of users: {}, Number of Movies: {}, Min rating: {}, Max rating: {}".format(
        num_users, num_movies, min_rating, max_rating
    )
)
print("\nEncoded Dataset:\n", df.head())

Number of users: 53, Number of Movies: 1595, Min rating: 1.0, Max rating: 5.0

Encoded Dataset:
    userId  movieId  rating  count  gender  user  movie
0     303      785     3.0    484       0     0      0
1     194      274     2.0    305       0     1      1
2     234     1184     2.0    480       0     2      2
3     308        1     4.0    397       0     3      3
4     181     1081     1.0    435       0     4      4


## Train & Test Split

In [20]:
df = df.sample(frac=1, random_state=42)
# extract input
x = df[["user", "movie"]].values

# xtract targets
yr = df["rating"].values
yg = df["gender"].values

# splitting
train_indices = int(0.9 * df.shape[0])
x_train, x_val = x[:train_indices], x[train_indices:]

In [21]:
# build the massive 7-lists together answer key for Multi-Task Learning!
y_train = [
    yr[:train_indices],                 # Target 1: Actual Rating
    yg[:train_indices],                 # Target 2: Gender
    yg[:train_indices],                 # Target 3: Gender
    yg[:train_indices],                 # Target 4: Gender
    yg[:train_indices],                 # Target 5: Gender
    np.zeros(train_indices),            # Target 6: Zeros
    np.zeros(train_indices)             # Target 7: Zeros
]

y_val = [
    yr[train_indices:],
    yg[train_indices:],
    yg[train_indices:],
    yg[train_indices:],
    yg[train_indices:],
    np.zeros(df.shape[0]-train_indices),
    np.zeros(df.shape[0]-train_indices)
]

print('x_train shape : ', x_train.shape)
print('x_val shape   : ', x_val.shape)

x_train shape :  (18674, 2)
x_val shape   :  (2075, 2)
